SECTION 1: SETUP AND LOAD PROCESSED DATA

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import gc


SECTION 2: CONFIGURATION

In [2]:
DATA_DIR = r"C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
OUTPUT_DIR = os.path.join(PROCESSED_DIR, 'features_parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input Directory: {PROCESSED_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

Input Directory: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data
Output Directory: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data\features_parquet


SECTION 3: LOAD DATA

In [3]:
print("\nLoading processed data...")
print("-" * 80)

df_ozone = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'ozone_processed.csv'),
    parse_dates=['DATE_TIME']
)
print(f"Loaded ozone data: {df_ozone.shape}")

# Load site metadata
try:
    df_site = pd.read_csv(os.path.join(PROCESSED_DIR, 'site_metadata.csv'))
    print(f"Loaded site metadata: {df_site.shape}")
except:
    df_site = pd.DataFrame()
    print("No site metadata available")


Loading processed data...
--------------------------------------------------------------------------------
Loaded ozone data: (21667030, 10)
Loaded site metadata: (159, 19)


SECTION 4: PREPROCESSING (DOWNCASTING)

In [ ]:
print("\nPreprocessing...")
print("-" * 80)

cols_to_drop = ['OZONE_F', 'UNITS', 'EXPECTED_VALUE', 'CALIBRATION_VALUE', 
                'CALIBRATION_TYPE', 'UPDATE_DATE']
df_ozone = df_ozone.drop(columns=[c for c in cols_to_drop if c in df_ozone.columns])
print(f"Dropped unnecessary columns. New shape: {df_ozone.shape}")

# Downcast to float32
float_cols = df_ozone.select_dtypes(include=['float64']).columns
df_ozone[float_cols] = df_ozone[float_cols].astype('float32')

int_cols = df_ozone.select_dtypes(include=['int64']).columns
df_ozone[int_cols] = df_ozone[int_cols].astype('int32')

print(f"Memory usage after downcast: {df_ozone.memory_usage(deep=True).sum() / 1e9:.2f} GB")



Preprocessing...
--------------------------------------------------------------------------------
Dropped unnecessary columns. New shape: (21667030, 4)
Memory usage after downcast: 1.54 GB


SECTION 5: FEATURE ENGINEERING

In [ ]:
def add_temporal_features(df):
    """Add time-based features to a pandas DataFrame"""
    df = df.copy()
    
    df['year'] = df['DATE_TIME'].dt.year.astype('int16')
    df['month'] = df['DATE_TIME'].dt.month.astype('int8')
    df['day'] = df['DATE_TIME'].dt.day.astype('int8')
    df['hour'] = df['DATE_TIME'].dt.hour.astype('int8')
    df['dayofweek'] = df['DATE_TIME'].dt.dayofweek.astype('int8')
    df['dayofyear'] = df['DATE_TIME'].dt.dayofyear.astype('int16')
    df['week'] = df['DATE_TIME'].dt.isocalendar().week.astype('int8')
    df['quarter'] = df['DATE_TIME'].dt.quarter.astype('int8')
    
    # Weekend indicator
    df['is_weekend'] = (df['dayofweek'] >= 5).astype('int8')
    
    # Rush hour indicator (7-9 AM and 5-7 PM)
    df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype('int8')
    
    # Season
    season_map = {
        12: 0, 1: 0, 2: 0,   # winter = 0
        3: 1, 4: 1, 5: 1,    # spring = 1
        6: 2, 7: 2, 8: 2,    # summer = 2
        9: 3, 10: 3, 11: 3   # fall = 3
    }
    df['season'] = df['month'].map(season_map).astype('int8')
    
    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7).astype('float32')
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7).astype('float32')
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25).astype('float32')
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25).astype('float32')
    
    return df

In [ ]:
def process_site_features(site_df):
    """
    Process all features for a single site.
    This function is applied to each site's data separately.
    """
    site_df = site_df.sort_values('DATE_TIME').copy()
    
    if 'OZONE' not in site_df.columns or site_df['OZONE'].isna().all():
        return site_df
    
    # LAG FEATURES 
    lags = [1, 2, 3, 6, 12, 24, 48, 168]
    for lag in lags:
        site_df[f'OZONE_lag_{lag}'] = site_df['OZONE'].shift(lag).astype('float32')
    
    # ROLLING FEATURES 
    windows = [3, 6, 12, 24, 168]
    for window in windows:
        rolling = site_df['OZONE'].rolling(window=window, min_periods=1)
        site_df[f'OZONE_rolling_mean_{window}'] = rolling.mean().astype('float32')
        site_df[f'OZONE_rolling_std_{window}'] = rolling.std().astype('float32')
        site_df[f'OZONE_rolling_min_{window}'] = rolling.min().astype('float32')
        site_df[f'OZONE_rolling_max_{window}'] = rolling.max().astype('float32')
        site_df[f'OZONE_rolling_range_{window}'] = (
            site_df[f'OZONE_rolling_max_{window}'] - site_df[f'OZONE_rolling_min_{window}']
        ).astype('float32')
    
    # DIFFERENCING FEATURES 
    periods = [1, 24, 168]
    for period in periods:
        site_df[f'OZONE_diff_{period}'] = site_df['OZONE'].diff(period).astype('float32')
    
    # RATE OF CHANGE 
    site_df['OZONE_velocity'] = site_df['OZONE'].diff(1).astype('float32')
    site_df['OZONE_acceleration'] = site_df['OZONE_velocity'].diff(1).astype('float32')
    
    # EXPANDING FEATURES
    site_df['OZONE_expanding_mean'] = site_df['OZONE'].expanding(min_periods=1).mean().astype('float32')
    site_df['OZONE_expanding_std'] = site_df['OZONE'].expanding(min_periods=1).std().astype('float32')
    
    # PERCENTILE FEATURES (site-level)
    p25 = site_df['OZONE'].quantile(0.25)
    p50 = site_df['OZONE'].quantile(0.50)
    p75 = site_df['OZONE'].quantile(0.75)
    p90 = site_df['OZONE'].quantile(0.90)
    
    site_df['OZONE_site_p25'] = p25
    site_df['OZONE_site_p50'] = p50
    site_df['OZONE_site_p75'] = p75
    site_df['OZONE_site_p90'] = p90
    site_df['OZONE_above_site_median'] = (site_df['OZONE'] > p50).astype('int8')
    site_df['OZONE_above_site_p75'] = (site_df['OZONE'] > p75).astype('int8')
    
    # INTERPOLATE MISSING VALUES
    numeric_cols = site_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if site_df[col].isna().any():
            site_df[col] = site_df[col].interpolate(
                method='linear', 
                limit=3, 
                limit_direction='both'
            )
    
    # CLIP OUTLIERS
    if site_df['OZONE'].notna().any():
        p01 = site_df['OZONE'].quantile(0.01)
        p99 = site_df['OZONE'].quantile(0.99)
        site_df['OZONE'] = site_df['OZONE'].clip(lower=p01, upper=p99)
    
    return site_df


SECTION 6: MAIN PROCESSING PIPELINE WITH DASK

In [ ]:
print("\n" + "=" * 80)
print("STARTING FEATURE ENGINEERING WITH DASK")
print("=" * 80)

# Step 1: Add temporal features
df_ozone = add_temporal_features(df_ozone)
print(f"   Added temporal features. Shape: {df_ozone.shape}")

# Step 2: Sort by site and time
df_ozone = df_ozone.sort_values(['SITE_ID', 'DATE_TIME']).reset_index(drop=True)

# Step 3: Convert to Dask DataFrame
n_partitions = max(1, len(df_ozone) // 500_000)
ddf = dd.from_pandas(df_ozone, npartitions=n_partitions)
print(f"   Created Dask DataFrame with {n_partitions} partitions")

del df_ozone
gc.collect()

# Step 4: Apply site-level feature engineering
meta_cols = {
    'SITE_ID': 'object',
    'DATE_TIME': 'datetime64[ns]',
    'OZONE': 'float32',
    'QA_CODE': 'int32',
    'year': 'int16',
    'month': 'int8',
    'day': 'int8',
    'hour': 'int8',
    'dayofweek': 'int8',
    'dayofyear': 'int16',
    'week': 'int8',
    'quarter': 'int8',
    'is_weekend': 'int8',
    'is_rush_hour': 'int8',
    'season': 'int8',
    'hour_sin': 'float32',
    'hour_cos': 'float32',
    'month_sin': 'float32',
    'month_cos': 'float32',
    'dayofweek_sin': 'float32',
    'dayofweek_cos': 'float32',
    'dayofyear_sin': 'float32',
    'dayofyear_cos': 'float32',
}

for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
    meta_cols[f'OZONE_lag_{lag}'] = 'float32'

for window in [3, 6, 12, 24, 168]:
    meta_cols[f'OZONE_rolling_mean_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_std_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_min_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_max_{window}'] = 'float32'
    meta_cols[f'OZONE_rolling_range_{window}'] = 'float32'

for period in [1, 24, 168]:
    meta_cols[f'OZONE_diff_{period}'] = 'float32'

meta_cols['OZONE_velocity'] = 'float32'
meta_cols['OZONE_acceleration'] = 'float32'
meta_cols['OZONE_expanding_mean'] = 'float32'
meta_cols['OZONE_expanding_std'] = 'float32'
meta_cols['OZONE_site_p25'] = 'float32'
meta_cols['OZONE_site_p50'] = 'float32'
meta_cols['OZONE_site_p75'] = 'float32'
meta_cols['OZONE_site_p90'] = 'float32'
meta_cols['OZONE_above_site_median'] = 'int8'
meta_cols['OZONE_above_site_p75'] = 'int8'

meta_df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in meta_cols.items()})

with ProgressBar():
    ddf_features = ddf.groupby('SITE_ID').apply(
        process_site_features,
        meta=meta_df
    ).reset_index(drop=True)

# Step 5: Merge with site metadata
if not df_site.empty:
    site_features = ['SITE_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 
                     'LAND_USE', 'TERRAIN', 'STATE']
    available_site_features = [f for f in site_features if f in df_site.columns]
    
    if available_site_features:
        df_site_subset = df_site[available_site_features]
        ddf_features = ddf_features.merge(df_site_subset, on='SITE_ID', how='left')
        print(f"   Merged {len(available_site_features)} site features")

# Step 6: Save to parquet
output_path = os.path.join(OUTPUT_DIR, 'features_engineered')

with ProgressBar():
    ddf_features.to_parquet(
        output_path,
        engine='pyarrow',
        compression='snappy',
        write_index=False
    )

print(f"   Saved to: {output_path}")



STARTING FEATURE ENGINEERING WITH DASK

Step 1: Adding temporal features...
   Added temporal features. Shape: (21667030, 23)

Step 2: Sorting data...

Step 3: Converting to Dask DataFrame...
   Created Dask DataFrame with 43 partitions

Step 4: Engineering site-level features (lag, rolling, diff, etc.)...
   This may take a while...

Step 5: Merging with site metadata...
   Merged 7 site features

Step 6: Saving to parquet...
[########################################] | 100% Completed | 57.41 s
   Saved to: C:\Users\snehi\OneDrive - California State University Chico\MATH699P\Data\processed_data\features_parquet\features_engineered


SECTION 7: VERIFICATION AND SUMMARY

In [ ]:
print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

# Load back a sample to verify
ddf_verify = dd.read_parquet(output_path)

print(f"\nFinal dataset info:")
print(f"   Total columns: {len(ddf_verify.columns)}")
print(f"   Total rows: {len(ddf_verify):,}")

print(f"\nColumn list:")
for i, col in enumerate(ddf_verify.columns):
    print(f"   {i+1:3d}. {col}")

# Show sample
print("\nSample of data:")
print(ddf_verify.head(10))

# Save feature list
feature_list_file = os.path.join(PROCESSED_DIR, 'feature_list_dask.txt')
with open(feature_list_file, 'w') as f:
    f.write("FEATURE LIST (Dask Processing)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total Features: {len(ddf_verify.columns)}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    for col in ddf_verify.columns:
        f.write(f"  • {col}\n")

print(f"\nFeature list saved to: {feature_list_file}")
print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE!")
print("=" * 80)


VERIFICATION

Loading sample to verify...

Final dataset info:
   Total columns: 75
   Total rows: 21,667,030

Column list:
     1. SITE_ID
     2. DATE_TIME
     3. OZONE
     4. QA_CODE
     5. year
     6. month
     7. day
     8. hour
     9. dayofweek
    10. dayofyear
    11. week
    12. quarter
    13. is_weekend
    14. is_rush_hour
    15. season
    16. hour_sin
    17. hour_cos
    18. month_sin
    19. month_cos
    20. dayofweek_sin
    21. dayofweek_cos
    22. dayofyear_sin
    23. dayofyear_cos
    24. OZONE_lag_1
    25. OZONE_lag_2
    26. OZONE_lag_3
    27. OZONE_lag_6
    28. OZONE_lag_12
    29. OZONE_lag_24
    30. OZONE_lag_48
    31. OZONE_lag_168
    32. OZONE_rolling_mean_3
    33. OZONE_rolling_std_3
    34. OZONE_rolling_min_3
    35. OZONE_rolling_max_3
    36. OZONE_rolling_range_3
    37. OZONE_rolling_mean_6
    38. OZONE_rolling_std_6
    39. OZONE_rolling_min_6
    40. OZONE_rolling_max_6
    41. OZONE_rolling_range_6
    42. OZONE_rolling_mean_12
